# 直接通过函数定义

In [6]:
from langchain_openai import ChatOpenAI

import os

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

In [1]:
from typing import Annotated, Any
from langchain_core.tools import tool


@tool(
    # 直接返回工具结果
    return_direct=True
)
def calculate(a: Annotated[float, "第一个数字"],
              b: Annotated[float, "第二个数字"],
              op: Annotated[str, "操作符(+ - * /)"]) -> float:
    """工具函数:计算两个数字的运算结果"""
    print(f"正在计算{a}和{b}的运算结果")
    return eval(f"{a} {op} {b}")


print(calculate.args_schema.model_json_schema())

{'description': '工具函数:计算两个数字的运算结果', 'properties': {'a': {'description': '第一个数字', 'title': 'A', 'type': 'number'}, 'b': {'description': '第二个数字', 'title': 'B', 'type': 'number'}, 'op': {'description': '操作符(+ - * /)', 'title': 'Op', 'type': 'string'}}, 'required': ['a', 'b', 'op'], 'title': 'calculate', 'type': 'object'}


# 定义参数类

In [2]:
from pydantic import BaseModel, Field


class CalculateArgs(BaseModel):
    a: float = Field(description="第一个数字")
    b: float = Field(description="第二个数字")
    op: str = Field(description="操作符(+ - * /)")


@tool(
    name_or_callable="calculate_v2",
    args_schema=CalculateArgs,
)
def calculate_v2(a: float, b: float, op: str) -> float:
    """
    工具函数:计算两个数字的运算结果
    """
    print(f"正在计算{a}和{b}的运算结果")
    return eval(f"{a} {op} {b}")


print(calculate_v2.args_schema.model_json_schema())

{'properties': {'a': {'description': '第一个数字', 'title': 'A', 'type': 'number'}, 'b': {'description': '第二个数字', 'title': 'B', 'type': 'number'}, 'op': {'description': '操作符(+ - * /)', 'title': 'Op', 'type': 'string'}}, 'required': ['a', 'b', 'op'], 'title': 'CalculateArgs', 'type': 'object'}


# Google风格注释定义

In [3]:
@tool(
    name_or_callable="calculate_v3",
    parse_docstring=True
)
def calculate_v3(a: float, b: float, op: str) -> float:
    """
    工具函数:计算两个数字的运算结果

    Args:
        a: 第一个数字
        b: 第二个数字
        op: 操作符(+ - * /)
    """

    print(f"正在计算{a}和{b}的运算结果")
    return eval(f"{a} {op} {b}")


print(calculate_v3.name)
print(calculate_v3.description)
print(calculate_v3.args_schema.model_json_schema())

calculate_v3
工具函数:计算两个数字的运算结果
{'description': '工具函数:计算两个数字的运算结果', 'properties': {'a': {'description': '第一个数字', 'title': 'A', 'type': 'number'}, 'b': {'description': '第二个数字', 'title': 'B', 'type': 'number'}, 'op': {'description': '操作符(+ - * /)', 'title': 'Op', 'type': 'string'}}, 'required': ['a', 'b', 'op'], 'title': 'calculate_v3', 'type': 'object'}


# 使用StructuredTool

In [4]:
from langchain_core.tools import StructuredTool


def calculate_v4(a: float, b: float, op: str) -> float:
    print(f"正在计算{a}和{b}的运算结果")
    return eval(f"{a} {op} {b}")


async def calculate_v4_async(a: float, b: float, op: str) -> float:
    print(f"正在计算{a}和{b}的运算结果")
    return eval(f"{a} {op} {b}")


# 创建了一个支持同步和异步的工具
calculate_struct = StructuredTool.from_function(
    # 同步函数
    func=calculate_v4,
    # 协程函数
    coroutine=calculate_v4_async,
    name="calculate_struct",
    description="计算两个数字的运算结果",
    return_direct=False,
)
calculate_struct

StructuredTool(name='calculate_struct', description='计算两个数字的运算结果', args_schema=<class 'langchain_core.utils.pydantic.calculate_struct'>, func=<function calculate_v4 at 0x000002BD08891F80>, coroutine=<function calculate_v4_async at 0x000002BD0AA32D40>)

# 将Runnable转化为Tool

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("""
帮我生成一个简短的，关于{topic}的报幕词，要求:
1. 内容搞笑一点
2. 输出内容采用{language}
""")
chain = prompt | model | StrOutputParser()
chain.invoke({"topic": "塑料袋", "language": "中文"})

'（轻快音乐骤停，换成悬疑紧张音效）\n\n（主持人故作神秘地压低声音）\n\n各位请注意！接下来要登场的这位，堪称“生活界的顶级特工”。\n\n它身披透明轻纱，看似弱不禁风。却拥有“一袋装万物，万物皆可装”的超能力——从两颗鸡蛋到十斤土豆，从外卖汤面到……不知道哪儿捡的石头。（耸肩）\n\n它神出鬼没，你家抽屉、门把手、甚至大衣口袋，都是它的秘密基地。当你终于决心“不再收集”，它总能在下一秒，带着超市Logo，再次闪亮回归！\n\n它寿命成谜——为你服务三分钟，却可能在自然环境中“续杯”几百年。这份执著，令人“感动”，更令人头皮发麻。\n\n让我们用热烈的掌声和一点点反思，有请——**塑料袋先生！** （可指向台下某个被风吹起的塑料袋，或工作人员举着的夸张巨型塑料袋道具）\n\n（音乐转为滑稽的进行曲）它来了它来了，它带着我们的生活与“白色烦恼”走来了！'

In [9]:
class ToolArgs(BaseModel):
    topic: str = Field(description="主题")
    language: str = Field(description="语言")


runnable_tool = chain.as_tool(
    name="generate_news",
    description="专门用于生成报幕词的工具",
    args_schema=ToolArgs,
)
runnable_tool.args_schema.model_json_schema()

{'properties': {'topic': {'description': '主题',
   'title': 'Topic',
   'type': 'string'},
  'language': {'description': '语言', 'title': 'Language', 'type': 'string'}},
 'required': ['topic', 'language'],
 'title': 'ToolArgs',
 'type': 'object'}

# 子类化BaseTool

In [11]:
from typing import Tuple, Union, List, Dict, Type
from langchain_core.tools import BaseTool
from langchain_community.tools import TavilySearchResults

# LangChain内置的搜索工具
search = TavilySearchResults(max_results=2, tavily_api_key=os.environ.get('TAVILY_API_KEY'))


class SearchArgs(BaseModel):
    query: str = Field(description="搜索内容")


class MySearchTool(BaseTool):
    name: str = "search_tool"
    description: str = "搜索互联网上公开的内容"
    return_direct: bool = False
    args_schema: Type[BaseModel] = SearchArgs

    def _run(self, query: str) -> Tuple[Union[List[Dict[str, str]], str], Dict]:
        return search.invoke({"query": query})


search_tool = MySearchTool()
print(search_tool.name)
print(search_tool.description)
print(search_tool.args_schema.model_json_schema())


search_tool
搜索互联网上公开的内容
{'properties': {'query': {'description': '搜索内容', 'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'SearchArgs', 'type': 'object'}
